# 01 · Generate synthetic claims data

Creates ~200 policyholders, ~300 pets/policies, 20 clinics, ~2,000 claims (~4–5% planted duplicates), and payments.
Also builds earned-premium and a small investment-holdings table for stretch Genie Code prompts.

`random.seed(42)` so row counts are stable across reruns.

**Run `00_setup` first.**


In [ ]:
%run ./00_setup


In [ ]:
import random
import datetime as dt
from pyspark.sql import Row

random.seed(42)
D = lambda s: dt.date.fromisoformat(s)

def rand_date(a, b):
    return a + dt.timedelta(days=random.randint(0, (b - a).days))

STATES = ["NC", "NY", "NJ", "CA", "TX", "FL", "GA", "IL", "PA", "OH"]
FIRST = ["Ava", "Liam", "Noah", "Emma", "Mia", "Owen", "Zoe", "Eli", "Ivy", "Leo", "Nora", "Jack"]
LAST = ["Smith", "Patel", "Garcia", "Kim", "Nguyen", "Brown", "Lopez", "Chen", "Davis", "Moore"]
DOGS = ["Labrador", "Golden Retriever", "French Bulldog", "Beagle", "Poodle", "Mixed"]
CATS = ["Domestic Shorthair", "Maine Coon", "Siamese", "Ragdoll", "Mixed"]
DIAG = {
    "Wellness": (80, 250),
    "Dental": (300, 1200),
    "GI Illness": (200, 1500),
    "Skin/Allergy": (120, 700),
    "Orthopedic": (800, 6000),
    "Cancer": (1500, 9000),
    "Accident/Injury": (300, 4000),
    "Urinary": (200, 1200),
}
CHANNELS = ["Mobile App", "Web Portal", "Email", "Clinic Direct"]
TIERS = {
    "Basic": (0.70, 500, 5000, 32),
    "Standard": (0.80, 250, 10000, 48),
    "Premium": (0.90, 100, 20000, 71),
}

policyholders = [
    Row(
        policyholder_id=f"PH{i:04d}",
        first_name=random.choice(FIRST),
        last_name=random.choice(LAST),
        email=f"user{i}@example.com",
        state=random.choice(STATES),
        signup_date=rand_date(D("2021-01-01"), D("2024-06-30")),
    )
    for i in range(1, 201)
]

clinics = [
    Row(
        clinic_id=f"VC{i:03d}",
        clinic_name=(
            f"{random.choice(['Oak', 'Pine', 'Maple', 'Cedar', 'River', 'Summit'])} "
            f"{random.choice(['Animal Hospital', 'Vet Clinic', 'Pet Care', 'Emergency Vets'])} {i}"
        ),
        city=random.choice(["Cary", "Raleigh", "Charlotte", "Durham", "Atlanta", "Austin"]),
        state=random.choice(STATES),
        is_emergency=(i % 5 == 0),
    )
    for i in range(1, 21)
]

pets, policies = [], []
for i in range(1, 301):
    ph = policyholders[(i - 1) % 200] if i <= 200 else random.choice(policyholders)
    sp = random.choice(["Dog", "Cat"])
    pets.append(
        Row(
            pet_id=f"PET{i:04d}",
            policyholder_id=ph.policyholder_id,
            pet_name=random.choice(["Max", "Bella", "Luna", "Charlie", "Milo", "Daisy", "Coco", "Rocky"]),
            species=sp,
            breed=random.choice(DOGS if sp == "Dog" else CATS),
            birth_date=rand_date(D("2012-01-01"), D("2023-06-01")),
        )
    )
    tier = random.choices(list(TIERS), [0.3, 0.5, 0.2])[0]
    r, ded, lim, prem = TIERS[tier]
    start = rand_date(D("2023-01-01"), D("2024-06-30"))
    policies.append(
        Row(
            policy_id=f"POL{i:04d}",
            pet_id=f"PET{i:04d}",
            policyholder_id=ph.policyholder_id,
            plan_tier=tier,
            annual_deductible=float(ded),
            reimbursement_rate=r,
            annual_limit=float(lim),
            monthly_premium=float(prem + random.randint(-5, 15)),
            start_date=start,
            end_date=start + dt.timedelta(days=730),
            policy_status=random.choices(["Active", "Cancelled"], [0.9, 0.1])[0],
        )
    )

print(len(policyholders), "policyholders,", len(pets), "pets,", len(policies), "policies,", len(clinics), "clinics")


In [ ]:
claims, payments = [], []
TODAY = D("2026-10-31")
n = 0

def make_claim(cid, pol, clinic_id, visit, diag, amt, inv, submitted, force_status=None, denial=None):
    status = force_status or random.choices(
        ["Paid", "Approved", "Denied", "In Review", "Submitted"],
        [0.62, 0.08, 0.15, 0.09, 0.06],
    )[0]
    decided = status in ("Paid", "Approved", "Denied")
    decision = submitted + dt.timedelta(days=random.randint(1, 21)) if decided else None
    if status == "Denied" and not denial:
        denial = random.choice(
            ["Pre-existing condition", "Not covered", "Missing documentation", "Waiting period"]
        )
    approved = None
    if decided:
        approved = 0.0 if status == "Denied" else round(amt * pol.reimbursement_rate, 2)
    c = Row(
        claim_id=cid,
        policy_id=pol.policy_id,
        pet_id=pol.pet_id,
        clinic_id=clinic_id,
        visit_date=visit,
        invoice_number=inv,
        diagnosis_category=diag,
        claimed_amount=float(amt),
        approved_amount=approved,
        claim_status=status,
        denial_reason=denial if status == "Denied" else None,
        submission_channel=random.choice(CHANNELS),
        submitted_date=submitted,
        decision_date=decision,
    )
    claims.append(c)
    if status == "Paid":
        payments.append(
            Row(
                payment_id=f"PAY{len(payments)+1:05d}",
                claim_id=cid,
                payment_date=decision + dt.timedelta(days=random.randint(1, 7)),
                payment_amount=approved,
                payment_method=random.choice(["Direct Deposit", "Check"]),
            )
        )
    return c

for _ in range(2000):
    n += 1
    pol = random.choice(policies)
    diag = random.choice(list(DIAG))
    lo, hi = DIAG[diag]
    visit = rand_date(D("2024-01-01"), D("2026-10-15"))
    orig = make_claim(
        f"CLM{n:06d}",
        pol,
        random.choice(clinics).clinic_id,
        visit,
        diag,
        round(random.uniform(lo, hi), 2),
        f"INV-{random.randint(100000, 999999)}",
        min(visit + dt.timedelta(days=random.randint(0, 10)), TODAY),
    )
    # ~4.5% planted double claims: same pet, clinic, visit date, NEW claim_id
    if random.random() < 0.045:
        n += 1
        style = random.choice(["same_inv", "suffix", "new_inv"])
        inv = {
            "same_inv": orig.invoice_number,
            "suffix": orig.invoice_number + "-R",
            "new_inv": f"INV-{random.randint(100000, 999999)}",
        }[style]
        amt = round(orig.claimed_amount * random.uniform(0.97, 1.03), 2)
        caught = random.random() < 0.35
        make_claim(
            f"CLM{n:06d}",
            pol,
            orig.clinic_id,
            visit,
            diag,
            amt,
            inv,
            min(orig.submitted_date + dt.timedelta(days=random.randint(3, 45)), TODAY),
            force_status="Denied" if caught else None,
            denial="Duplicate claim" if caught else None,
        )
    # Trap: sibling pet, same day, is NOT a duplicate
    if random.random() < 0.01:
        sib = [p for p in policies if p.policyholder_id == pol.policyholder_id and p.pet_id != pol.pet_id]
        if sib:
            n += 1
            make_claim(
                f"CLM{n:06d}",
                sib[0],
                orig.clinic_id,
                visit,
                "Wellness",
                round(random.uniform(80, 250), 2),
                f"INV-{random.randint(100000, 999999)}",
                orig.submitted_date,
            )

tables = {
    "policyholders": policyholders,
    "vet_clinics": clinics,
    "pets": pets,
    "policies": policies,
    "claims": claims,
    "claim_payments": payments,
}
for name, rows in tables.items():
    spark.createDataFrame(rows).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(name)
    print(name, len(rows))


In [ ]:
# Earned premium (monthly) + a small investment book for stretch prompts
earned = []
for p in policies:
    m = p.start_date.replace(day=1)
    while m <= min(p.end_date, TODAY):
        earned.append(Row(policy_id=p.policy_id, plan_tier=p.plan_tier, month=m, earned_premium=p.monthly_premium))
        m = (m.replace(day=28) + dt.timedelta(days=4)).replace(day=1)
spark.createDataFrame(earned).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("earned_premium_monthly")
print("earned_premium_monthly", len(earned))

ASSET = {
    "Corporate Bond": (0.045, 6.5),
    "Treasury": (0.039, 8.0),
    "Municipal Bond": (0.033, 7.2),
    "Agency MBS": (0.051, 5.1),
    "Commercial Mortgage": (0.058, 4.4),
    "Private Credit": (0.087, 3.2),
    "Public Equity": (0.0, 0.0),
}
RATING = ["AAA", "AA", "A", "BBB", "BB"]
holdings = []
for i in range(1, 401):
    a = random.choice(list(ASSET))
    y, dur = ASSET[a]
    holdings.append(
        Row(
            holding_id=f"H{i:04d}",
            asset_class=a,
            sector=random.choice(["Financials", "Utilities", "Industrials", "Healthcare", "Tech", "Government", "Real Estate"]),
            rating=None if a == "Public Equity" else random.choice(RATING),
            market_value=round(random.uniform(1e6, 5e7), 2),
            book_yield=round(max(y + random.uniform(-0.01, 0.01), 0), 4),
            duration_years=round(max(dur + random.uniform(-2, 2), 0), 2),
            as_of_date=D("2026-09-30"),
        )
    )
spark.createDataFrame(holdings).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("investment_holdings")
print("investment_holdings", len(holdings))


## Table and column comments — this is what makes Genie trustworthy


In [ ]:
# Comments are applied in the current catalog.schema
comments = [
    "COMMENT ON TABLE claims IS 'One row per submitted claim. Contains some duplicate submissions for the same pet, clinic, and visit date.'",
    "ALTER TABLE claims ALTER COLUMN claim_id COMMENT 'Unique claim ID. A resubmission of the same visit gets a NEW claim_id.'",
    "ALTER TABLE claims ALTER COLUMN invoice_number COMMENT 'Clinic invoice number as submitted; resubmissions may reuse or alter it. Do not treat as a unique key.'",
    "ALTER TABLE claims ALTER COLUMN claimed_amount COMMENT 'Amount billed by the clinic, USD.'",
    "ALTER TABLE claims ALTER COLUMN approved_amount COMMENT 'Reimbursable amount after reimbursement rate; 0 when denied; NULL while pending.'",
    "ALTER TABLE claims ALTER COLUMN claim_status COMMENT 'Submitted, In Review, Approved, Denied, or Paid.'",
    "ALTER TABLE claims ALTER COLUMN denial_reason COMMENT 'Populated when claim_status = Denied. Use Duplicate claim for caught resubmissions.'",
    "ALTER TABLE claims ALTER COLUMN pet_id COMMENT 'The pet treated on this visit. Duplicates are same pet_id + clinic_id + visit_date.'",
    "COMMENT ON TABLE claim_payments IS 'Payouts issued for claims with status Paid. Join to claims on claim_id.'",
    "COMMENT ON TABLE policies IS 'One row per pet insurance policy. reimbursement_rate is a decimal (0.80 = 80%).'",
    "ALTER TABLE policies ALTER COLUMN reimbursement_rate COMMENT 'Share reimbursed, decimal (0.80 = 80%). Display as a percentage.'",
    "COMMENT ON TABLE pets IS 'Insured pets. Multiple pets can share a policyholder_id (household).'",
    "COMMENT ON TABLE policyholders IS 'Customers. policyholder_id is the household key.'",
    "COMMENT ON TABLE vet_clinics IS 'Clinics that submitted claims.'",
    "COMMENT ON TABLE earned_premium_monthly IS 'Earned premium by policy and month. Use with claim_payments for loss ratio.'",
    "COMMENT ON TABLE investment_holdings IS 'Synthetic portfolio snapshot as of 2026-09-30. Stretch content only.'",
]
for stmt in comments:
    spark.sql(stmt)
print("Comments applied.")


## Answer key (facilitators) — print after generate


In [ ]:
spark.sql("""
CREATE OR REPLACE VIEW v_suspected_duplicate_claims AS
WITH ranked AS (
  SELECT c.*,
    ROW_NUMBER() OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS visit_rank,
    FIRST_VALUE(claim_id) OVER (PARTITION BY pet_id, clinic_id, visit_date ORDER BY submitted_date, claim_id) AS original_claim_id
  FROM claims c
)
SELECT * FROM ranked WHERE visit_rank > 1
""")

display(spark.sql("""
SELECT
  COUNT(*) AS suspected_duplicates,
  COUNT_IF(d.claim_status = 'Paid') AS duplicates_paid,
  COUNT_IF(d.claim_status = 'Denied' AND d.denial_reason = 'Duplicate claim') AS duplicates_caught,
  ROUND(SUM(p.payment_amount), 2) AS leakage_usd
FROM v_suspected_duplicate_claims d
LEFT JOIN claim_payments p USING (claim_id)
"""))
